# Module 08 — Pearson Correlation & Partial Correlation
Dedicated notebook for Module 08 — reproduces Pearson r + Spearman rho (classroom data) AND partial correlation (real PISA data) — the partial-correlation part is NOT supported by PSPP 2.1.1 (`PARTIAL CORR is not yet implemented`), it only runs here.

**This notebook is written to be self-explanatory without the web page** — every method includes a "what question does this answer" section before the code, and a "what does the result mean" section after it.

Human-reviewed English translation of the original Vietnamese notebook (code and results are identical, only text is translated).

## 0. What question does Pearson correlation answer?
The question the Pearson coefficient (r) answers is: **"Do these two variables tend to rise/fall together along a STRAIGHT line, and how strongly?"** Nothing more, nothing less. Specifically:

- **Sign (+/−)**: move together (positive) or in opposite directions (negative).
- **Magnitude |r|** (0 to 1): how well a straight line fits the cloud of data points — NOT "how strongly A influences B".
- **r says NOTHING about causation** — it only describes a co-occurring linear pattern. A high r could mean A causes B, B causes A, a third variable C causes both, or pure chance (small sample).
- **The property that matters most for the rest of this notebook: r is SYMMETRIC** — r(study_hours, math_score) = r(math_score, study_hours), the same number regardless of which direction you compute it. This is where confusion often creeps in once you start talking about "if X rises by 1, how much does Y rise" — see section 5 for why.

In [ ]:
# Run this cell first (Colab: click ▶). No installation needed.
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
sns.set_theme(style="whitegrid"); plt.rcParams["figure.figsize"] = (7, 4)
pd.set_option("display.precision", 3)

def make_data(seed=2026, n=240):
    """SIMULATED 'Classroom of 240 students' dataset — SAME seed=2026 used
    throughout Modules 03-08, exactly matching lop_hoc_240.csv/.sav in the
    repo (not real data, but 100% reproducible)."""
    rng = np.random.default_rng(seed)
    school = rng.choice(list("ABC"), n, p=[.35, .35, .30])
    gender = rng.choice(["Male", "Female"], n)
    method = rng.choice(["Traditional", "Project-based"], n)
    study_hours = np.clip(rng.gamma(4, 1.2, n), 0.5, 15).round(1)
    interest = rng.normal(0, 1, n) + 0.15 * (method == "Project-based")
    anxiety = rng.normal(0, 1, n) - 0.25 * interest
    def likert(lat, load, noise=0.7):
        return np.clip(np.round(3 + load * lat + rng.normal(0, noise, n)), 1, 5).astype(int)
    d = pd.DataFrame({"id": np.arange(1, n + 1), "school": school, "gender": gender, "method": method,
                      "study_hours": study_hours})
    for k, (lat, load) in enumerate([(interest, .8), (interest, .7), (interest, .75)], 1):
        d[f"h{k}"] = likert(lat, load)
    for k, (lat, load) in enumerate([(anxiety, .8), (anxiety, .75), (anxiety, .7)], 1):
        d[f"a{k}"] = likert(lat, load)
    eff = d.school.map({"A": 3, "B": 0, "C": -3}).to_numpy()
    d["pretest"] = (rng.normal(60, 10, n) + eff).round(1)
    d["posttest"] = np.clip(d.pretest + 3 + 5 * (method == "Project-based") + rng.normal(0, 6, n), 0, 100).round(1)
    d["math"] = np.clip(35 + 2.5 * study_hours + 4 * interest - 3 * anxiety + eff + rng.normal(0, 6, n), 0, 100).round(1)
    d.loc[6, "study_hours"] = 48.0
    d.loc[[11, 57, 130], "h2"] = np.nan
    d["h2"] = d["h2"].astype("Int64")
    return d

df = make_data()
print("Created df:", df.shape)

## 1. Clean the `study_hours` outlier (same as Modules 04/05)

In [ ]:
mask = df["study_hours"] > 20
df.loc[mask, "study_hours"] = df.loc[mask, "study_hours"] / 10

## 2. Pearson r: study hours vs Math score

In [ ]:
r, p = stats.pearsonr(df["study_hours"], df["math"])
print(f"Pearson r={r:.3f}, r²={r**2:.3f}, p={p:.2e}")

#### 📤 Actual output
r≈0.582, r²≈0.338 (explains ≈34% of the variance in Math score), p<0.001. ✅ Matches the lesson.

### ❓ A common question: "If study hours go up by 1, how much does Math score go up — and conversely, if Math score goes up by 1, how much do study hours go up?"
**No, that's not how it works.** r only tells you the STRENGTH of the linear relationship — it does not answer "how much does Y change per 1-unit change in X". That is the job of the REGRESSION SLOPE (b), a DIFFERENT quantity that is **not symmetric**. Formula: b(Y on X) = r × (SD of Y / SD of X). Because SD(Y)/SD(X) differs from SD(X)/SD(Y) (unless the two variables happen to share the same SD), the two regression directions give DIFFERENT numbers — demonstrated with real figures right below.

In [ ]:
sx = df["study_hours"].std(ddof=1); sy = df["math"].std(ddof=1)
b_math_on_hours = r * sy / sx   # 1 more study hour -> how much does Math score change
b_hours_on_math = r * sx / sy   # 1 more Math point -> how much do study hours change
print(f"SD(study hours)={sx:.3f}  SD(Math score)={sy:.3f}")
print(f"+1 study hour/week -> Math score changes by {b_math_on_hours:.2f} points on average")
print(f"+1 Math point -> study hours/week change by {b_hours_on_math:.3f} hours (~{b_hours_on_math*60:.0f} minutes) on average")

#### 📤 Actual output — this is the direct answer to "how do the two directions compare?"
+1 study hour/week → Math score rises by ≈2.51 points on average. But +1 Math point → study hours rise by only ≈0.135 hours (~8 minutes) on average. These two numbers are NOT reciprocals of each other, and the gap between them is large (2.51 vs 0.135) — because the two variables have completely different units (hours vs points) AND different amounts of spread (SD). r=0.582 is the ONE number that stays the same regardless of direction; "if X goes up by 1, how much does Y go up" always requires specifying WHICH variable is being treated as the dependent one.

## 3. Spearman rho — compared against Pearson

In [ ]:
rho, p_rho = stats.spearmanr(df["study_hours"], df["math"])
print(f"Spearman rho={rho:.3f}, p={p_rho:.2e}")

#### 📤 Actual output
rho≈0.554 — fairly close to Pearson r since the relationship is reasonably linear. **How is this method different from Pearson?** Spearman rho answers the SAME question (do the two variables tend to move together) but uses RANKS instead of raw values — so it doesn't require a perfectly straight-line relationship, only a "monotonic" one (as one variable rises, the other tends to rise too, not necessarily at a fixed rate). Use it when you suspect strong outliers or a curved-but-monotonic relationship.

In [ ]:
fig, ax = plt.subplots()
sns.regplot(data=df, x="study_hours", y="math", ax=ax, scatter_kws={"alpha":.5})
ax.set_title("Weekly study hours vs Math score (n=240, after cleaning 1 outlier)")
plt.show()

## 4. Partial correlation — real PISA data (195 schools)
**What question does this method answer?** Partial correlation answers: "If we 'hold constant' the influence of a third variable Z, how much of the relationship between X and Y remains?" — used when you suspect part (or all) of the observed X-Y correlation actually comes from both being related to Z, rather than X and Y being directly related.

Fetch the school-level dataset directly from GitHub — variables `EDUSHORT` (educational material shortage), `sci_mean` (school mean Science score), `STAFFSHORT` (staff shortage, the suspected confounder).

In [ ]:
RAW = "https://raw.githubusercontent.com/TatcataiTTN/for-Social-Science/main/SPSS/data/sav/"
truong = pd.read_csv(RAW + "vnm_truong_195_tong_hop.csv")
d = truong[["EDUSHORT", "sci_mean", "STAFFSHORT"]].dropna()
print("n =", len(d))

r_xy = stats.pearsonr(d["EDUSHORT"], d["sci_mean"])[0]
r_xz = stats.pearsonr(d["EDUSHORT"], d["STAFFSHORT"])[0]
r_yz = stats.pearsonr(d["sci_mean"], d["STAFFSHORT"])[0]
r_partial = (r_xy - r_xz*r_yz) / np.sqrt((1-r_xz**2)*(1-r_yz**2))
print(f"r(EDUSHORT, Science score) raw = {r_xy:.3f}")
print(f"r(EDUSHORT, STAFFSHORT)        = {r_xz:.3f}")
print(f"r(Science score, STAFFSHORT)   = {r_yz:.3f}")
print(f"Partial correlation            = {r_partial:.3f}")

#### 📤 Actual output
Raw correlation EDUSHORT × Science score: r=-0.039 (very weak). r(EDUSHORT, STAFFSHORT)≈0.61 (fairly high — the suspected confounder). r(Science score, STAFFSHORT)≈0.002 (nearly 0). Partial correlation after controlling for STAFFSHORT: r=-0.051 — **barely changes**.

**Why does it barely change?** The partial-correlation formula only shifts substantially when the control variable Z is genuinely correlated with BOTH main variables. Here, STAFFSHORT is barely related to the Science score (r≈0.002) — so "removing" it doesn't change the original picture much.

## 5. Full demonstration: the ASYMMETRY of regression — EDULEAD ↔ NEGSCLIM
This is a direct answer to the question: *"If a 1-point rise in Educational Leadership lowers Negative School Climate by 0.06 points, does a 1-point rise in Negative School Climate lower Educational Leadership by something HUGE (like 'a dozen-plus points')?"*

**Short answer: NO — but the two directions ARE different, just by a ratio that follows a precise formula, not something random or "inverse-amplified".**

In [ ]:
d2 = truong[["EDULEAD", "NEGSCLIM"]].dropna()
r2, p2 = stats.pearsonr(d2["EDULEAD"], d2["NEGSCLIM"])
sx2 = d2["EDULEAD"].std(ddof=1); sy2 = d2["NEGSCLIM"].std(ddof=1)
b_neg_on_edu = r2 * sy2 / sx2   # +1 EDULEAD -> how much does NEGSCLIM change
b_edu_on_neg = r2 * sx2 / sy2   # +1 NEGSCLIM -> how much does EDULEAD change
print(f"n={len(d2)}, r={r2:.4f}, p={p2:.4f}  (p>0.05 -> NOT statistically significant, see the warning below)")
print(f"SD(EDULEAD)={sx2:.4f}  SD(NEGSCLIM)={sy2:.4f}")
print(f"+1 point of EDULEAD  -> NEGSCLIM changes by {b_neg_on_edu:+.4f} points on average")
print(f"+1 point of NEGSCLIM -> EDULEAD changes by  {b_edu_on_neg:+.4f} points on average")
print(f"Ratio of the two slopes = {b_neg_on_edu/b_edu_on_neg:.3f}  =  (SD_NEGSCLIM / SD_EDULEAD)² = {(sy2/sx2)**2:.3f}")

#### 📤 Actual output — and how to read it correctly
r=-0.057, **p=0.427 → NOT statistically significant** (this relationship is weak enough that it could just be sampling noise — shown here PURELY to illustrate the mathematics of asymmetry, NOT to claim "better leadership reduces a negative climate" — with this p-value there isn't enough evidence for that).

On the arithmetic of "how do the two directions compare": +1 point of EDULEAD → NEGSCLIM changes by **-0.085 points** on average. +1 point of NEGSCLIM → EDULEAD changes by **-0.038 points** on average. These two numbers differ by **exactly 2.21x** — and 2.21 is exactly (SD of NEGSCLIM / SD of EDULEAD)². **Not "a dozen-plus times"** as intuition sometimes guesses — the gap between the two regression directions is always bounded by the SQUARED RATIO OF THE STANDARD DEVIATIONS of the two variables, not an arbitrary number.

**General rule to remember:**
1. Correlation r: **one single number**, symmetric, direction-independent.
2. Regression slope b: **two different numbers** depending on which variable is treated as dependent — formula: b(Y on X) = r×SD(Y)/SD(X).
3. The product of the two slopes always equals r²: b(Y on X) × b(X on Y) = r² — a quick way to check whether two slopes were computed correctly.
4. The two slopes are equal ONLY when SD(X) = SD(Y) — rare when two variables are measured on different scales.

In [ ]:
# Verify the "product of slopes = r-squared" identity
print(f"b_yx * b_xy = {b_neg_on_edu * b_edu_on_neg:.5f}")
print(f"r^2         = {r2**2:.5f}")
print("These two match -> confirms the formula is correct, not a coincidence.")